In [3]:
import os
import re
import math
import shutil
import subprocess

from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("QtAgg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from matplotlib.dates import DateFormatter
from dateutil.tz import tzutc, tzlocal
from scipy import stats
from scipy.optimize import brentq, curve_fit, fsolve


import sys
import signal
import tempfile
import time
from collections import Counter
from IPython.display import clear_output

from scipy.interpolate import PchipInterpolator
from scipy.interpolate import UnivariateSpline

In [4]:
%load_ext autoreload
%autoreload 2

import GSSHA_set_and_run_functions as gf

%load_ext autoreload
%autoreload 2

import GSSHA_post_process_functions as gppf

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Paths and iteration files for running all the model simulations

paths that don't change

In [5]:
cwd = os.getcwd()
script_dir = Path.cwd() 
GSSHA_executables = script_dir / 'GSSHA_executables'

GSSHA Model Name and iteration file input

In [6]:
GSSHA_prj_name = "Waialua_FIM_testing"
model_dir = script_dir / GSSHA_prj_name

xys_tsf_iteration_file_name = GSSHA_prj_name + "_ITERATION"
xys_tsf_iteration_value = "155.0"

# Force shutdown of any GSSHA model running in model folder

In [7]:
 # Use  if you're in Jupyter or interactive session
process = gf.GSSHA_auto_shutdown(model_dir)
gf.force_shutdown_gssha(process=process, model_dir=model_dir)

No gssha.exe found. The model folder may already be clean.
GSSHA stopped and gssha.exe is unlocked.


True

# 1.0 Run flow testing for model development

Set up test flows and total modeling time (see example of Waialua below)

In [8]:
FLOW_TEST = [2]
tot_time = 1000
test_input_flow_results_dir = model_dir / "TEST_INPUT_FLOW_SIMULATIONS"

Run test flows in model with total time unchanged for each flow (be reasonable on high test flows, it is not necessary to reach the max flood stage yet due to computation requirements of those runs.

A window will pop up where you can type q to exit the simulation. ONLY type q if the model looks like it has converged. A dataframe will pop upshowing the changes in the flood through each timestep. If the model run time is excessively long, evaluate the dataframe and decide if the model has converged.   

In [9]:
gf.cleanup_model_dir(model_dir)

for flow in FLOW_TEST:
    new_value = float(flow)
    cfs_flow_for_filenames = round(flow*35.31467)
    
    output_file = (
        test_input_flow_results_dir
        / f"{GSSHA_prj_name}_OUTPUT-input-flow-cfs-{cfs_flow_for_filenames}.tsf"
    )
    
    if output_file.exists():
        print(f"{output_file.name} already exists. Skipping...")
        continue

    
    df_prj = gf.read_prj_file(GSSHA_prj_name +".prj", prj_folder_path = model_dir)
    

    #ITERATION FILES
    gf.replace_value_in_gssha_file(read_dir=model_dir , gssha_sample_file = xys_tsf_iteration_file_name + ".tsf", save_filename = GSSHA_prj_name + "_bc.tsf", old_value =xys_tsf_iteration_value, new_value = new_value)
    gf.replace_value_in_gssha_file(read_dir= model_dir, gssha_sample_file = xys_tsf_iteration_file_name + ".xys", save_filename = GSSHA_prj_name + ".xys", old_value = xys_tsf_iteration_value, new_value = new_value)
    ##UPDATE THE ITERATION W THE NEW XYS AND TSF FILE

    #!!!!write a function to copy with specified cfs flows in name!!!

    #SAVE TEXT XYS file for knowing the input flows for model run
    #These files are for the 

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + "_bc.tsf",
        output_folder=test_input_flow_results_dir,
        new_filename=GSSHA_prj_name + "_OUTPUT-input-flow-cfs-" + str(cfs_flow_for_filenames) + ".tsf"
    )


    

    df_prj.loc['TOT_TIME', 'Value'] = tot_time
    
    df_prj = df_prj.reset_index()


    #UPDATE PRJ FILE WHICH CONTAINS THE PATHS
    gf.convert_df_to_prj(
        df_prj,
        output_folder= model_dir,
        prj_file_name=GSSHA_prj_name
    )


    gf.copy_gssha_apps_to_model(GSSHA_executables, model_dir)
    
    #this function allows you to see when model convergence occurs so you can quit the run
    return_code = gf.run_gssha_convergence_view(
    MODEL_DIR=model_dir,
    PROJECT_FILE=GSSHA_prj_name + ".prj",
    DEP_FILE=GSSHA_prj_name + ".dep",
    cell_size=10,
    dep_check_seconds=15,
    display_last_n=10
)
    # move_and_rename_gssha_output(MODEL_DIR, RESULTS_DIR, output_description = "TEST_RUN" + first_date, extension = "otl")
    gf.cleanup_model_dir(model_dir)

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".dep",
        output_folder=test_input_flow_results_dir,
        new_filename=GSSHA_prj_name + "_OUTPUT-timeseries-depth-m-" + str(cfs_flow_for_filenames) + ".dep"
    )

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".gfl",
        output_folder=test_input_flow_results_dir,
        new_filename=GSSHA_prj_name + "_OUTPUT-maxflood-dep-m-" + str(cfs_flow_for_filenames) + ".gfl"
    )
    
    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".oqc",
        output_folder=test_input_flow_results_dir,
        new_filename=GSSHA_prj_name + "_OUTPUT_USGS-STATS-location-cms-" + str(cfs_flow_for_filenames) + ".oqc"
    )
    
    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".ows",
        output_folder=test_input_flow_results_dir,
        new_filename=GSSHA_prj_name + "_OUTPUT_WSE-active-USGS-gauge-m-" + str(cfs_flow_for_filenames) + ".ows"
    )


✔️ Cleaned up model folder.
Waialua_FIM_testing_OUTPUT-input-flow-cfs-71.tsf already exists. Skipping...


# 2.0 Read TEST FLOW results, calibrate for initial flood stage model inputs

Set up gauge datum offset and min/max flood stages

In [10]:
Gauge_Datum_offset = 13.835
#offset between LMSL and gauge recordings

#create directory to store model output and the input parameters
calibration_one_results_dir = model_dir / ("CALIBRATION_STAGE_MAPS_WITH_OFFSET_" + str(Gauge_Datum_offset))
calibration_one_results_dir.mkdir(parents=True, exist_ok=True)

Min_flood_stage = 26 
#minor flood stage from NWS (of 1-2 feet lower if you prefer)

Max_flood_stage = 34
#max recorded gauge level or NWS major reported stage (or 1-3 feet over if you have ample computer time, a couple feet higher is better for calibration)

Create a dataframe to store results. we will use power law curves to derive connections between the input flows and gauge levels in the model. Then we will evaluate convergence thresholds to ensure run time is not excessively long. 

In [11]:
#SET UP CALIBRATION FROM INITIAL INPUT FLOWS

# Once Offset is determined, set up model paramters for running stage height maps
gage_readings = np.arange(
    Min_flood_stage,
    Max_flood_stage + 1, 
    1,
).tolist()

#Set up the storage of results
modeling_test_results= {"gauge_readings_ft":[],
                            "expected_WSE_m":[],
                            "expected_WSE_ft":[],  
                            "convergence_time": [], 
                            "input_flows":[]}

#Fill in some metadata about gauge readings, expected model outputs to figure out the rest
expected_model_outputs = [x - Gauge_Datum_offset for x in gage_readings]
modeling_test_results["gauge_readings_ft"] = gage_readings
modeling_test_results["expected_WSE_ft"] = expected_model_outputs
modeling_test_results["expected_WSE_m"]  = [round(ft * 0.3048, 2) for ft in expected_model_outputs]

MODEL INPUT FLOW FOR EXPECTED WSE

In [13]:
modeling_test_results["input_flows"] = gppf.get_input_flows_for_gauge_stages(
    modeling_results_directory = test_input_flow_results_dir,
    requested_WSE =  modeling_test_results["expected_WSE_m"],
    plot=True,
)

MODEL TIMESTEP CONVERGENCE (YOU CHANGE PARAMETERS BASED ON GRAPH)

In [14]:
gppf.view_timestep_convergence(model_results_directory = test_input_flow_results_dir, area_change_threshold = 1000, smoothing=0.20, number_timesteps_below_threshold = 6)

{'function': <scipy.interpolate._fitpack2.UnivariateSpline at 0x1e9230b1460>,
 'smoothing': 0.2,
 'spline_s': 179640.00000000003,
 'min_flow': 2.0,
 'max_flow': 1000.0,
 'flow': array([   2.,    5.,   10.,   20.,   30.,   40.,   60.,   80.,   90.,
         100.,  120.,  140.,  150.,  160.,  180.,  200.,  240.,  280.,
         300.,  400.,  600., 1000.]),
 'time': array([420.5, 390.5, 390.5, 360.5, 330.5, 720.5, 390.5, 750.5, 690.5,
        660.5, 990.5, 900.5, 690.5, 660.5, 810.5, 840.5, 480.5, 540.5,
        450.5, 390.5, 390.5, 300.5])}

In [15]:
medium_flow_threshold = 400
high_flow_threshold = 1000

low_flow_timing = 1000
medium_flow_timing = 420
high_flow_timing = 330


modeling_test_results["convergence_time"] = gppf.predict_timestep_convergence(
    modeling_test_results["input_flows"],
    medium_flow_threshold,
    high_flow_threshold,
    low_flow_timing,
    medium_flow_timing,
    high_flow_timing,
)

Save calibration one to new folder so we can re run the model with a better fit

In [16]:
#save the model input parameters (from the previous section, derived from test flow inputs)
first_calibration_model_input = pd.DataFrame(modeling_test_results).set_index('gauge_readings_ft')
first_calibration_model_input.to_csv(calibration_one_results_dir / "initial_stage_maps_input_flows_from_first_calibration.csv")
print(calibration_one_results_dir)

C:\Users\bgorberg\Documents\GitHub\Flood_Stage_Maps\RUN_GSSHA\Waialua_FIM_testing\CALIBRATION_STAGE_MAPS_WITH_OFFSET_13.835


In [52]:
first_calibration_model_input

,expected_WSE_m,expected_WSE_ft,convergence_time,input_flows
gauge_readings_ft,,,,
26,3.71,12.165,1000,78.798538
27,4.01,13.165,1000,110.943726
28,4.32,14.165,1000,157.068362
29,4.62,15.165,1000,218.588602
30,4.93,16.165,1000,305.575533
31,5.23,17.165,420,419.737594
32,5.54,18.165,420,578.323068
33,5.84,19.165,420,782.521683
34,6.15,20.165,330,1060.448277


# 2.1 Run the model using the initial stage map parameter input (determined from the test flow simulations)

Create folder to store model input parametes and initial stage map results

In [17]:
#create directory to store model output and the input parameters
calibration_one_results_dir = model_dir / ("CALIBRATION_STAGE_MAPS_WITH_OFFSET_" + str(Gauge_Datum_offset))
calibration_one_results_dir.mkdir(parents=True, exist_ok=True)

#save the model input parameters (from the previous section, derived from test flow inputs)
first_calibration_model_input = pd.DataFrame(modeling_test_results).set_index('gauge_readings_ft')
first_calibration_model_input.to_csv(calibration_one_results_dir / "initial_stage_maps_input_flows_from_first_calibration.csv")
print(calibration_one_results_dir)

C:\Users\bgorberg\Documents\GitHub\Flood_Stage_Maps\RUN_GSSHA\Waialua_FIM_testing\CALIBRATION_STAGE_MAPS_WITH_OFFSET_13.835


In [18]:
first_calibration_model_input

,expected_WSE_m,expected_WSE_ft,convergence_time,input_flows
gauge_readings_ft,,,,
26,3.71,12.165,1000,78.798538
27,4.01,13.165,1000,110.943726
28,4.32,14.165,1000,157.068362
29,4.62,15.165,1000,218.588602
30,4.93,16.165,1000,305.575533
31,5.23,17.165,420,419.737594
32,5.54,18.165,420,578.323068
33,5.84,19.165,420,782.521683
34,6.15,20.165,330,1060.448277


Re run the model using the input parameters and save results to the directory specified above

In [19]:
gf.cleanup_model_dir(model_dir)

for gauge_reading in first_calibration_model_input.index:
    input_flow = first_calibration_model_input['input_flows'][gauge_reading].round(0)
    total_run_time = first_calibration_model_input['convergence_time'][gauge_reading]

    
    new_value = float(input_flow)
    cfs_flow_for_filenames = round(input_flow*35.31467)
    
    output_file = (
        calibration_one_results_dir
        / f"{GSSHA_prj_name}_{gauge_reading}_OUTPUT-input-flow-cfs-{cfs_flow_for_filenames}.tsf"
    )
    
    if any(
        str(gauge_reading) in file.name
        for file in calibration_one_results_dir.iterdir()
    ):
        print(f"File containing {gauge_reading} already exists. Skipping...")
        continue

    
    df_prj = gf.read_prj_file(GSSHA_prj_name +".prj", prj_folder_path = model_dir)
    

    #ITERATION FILES
    gf.replace_value_in_gssha_file(read_dir=model_dir , gssha_sample_file = xys_tsf_iteration_file_name + ".tsf", save_filename = GSSHA_prj_name + "_bc.tsf", old_value =xys_tsf_iteration_value, new_value = new_value)
    gf.replace_value_in_gssha_file(read_dir= model_dir, gssha_sample_file = xys_tsf_iteration_file_name + ".xys", save_filename = GSSHA_prj_name + ".xys", old_value = xys_tsf_iteration_value, new_value = new_value)
    ##UPDATE THE ITERATION W THE NEW XYS AND TSF FILE

    #!!!!write a function to copy with specified cfs flows in name!!!

    #SAVE TEXT XYS file for knowing the input flows for model run
    #These files are for the 

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + "_bc.tsf",
        output_folder=calibration_one_results_dir,
        new_filename=GSSHA_prj_name +"_" + str(gauge_reading) + "_OUTPUT-input-flow-cfs-" + str(cfs_flow_for_filenames) + ".tsf"
    )


    

    df_prj.loc['TOT_TIME', 'Value'] = total_run_time
    
    df_prj = df_prj.reset_index()


    #UPDATE PRJ FILE WHICH CONTAINS THE PATHS
    gf.convert_df_to_prj(
        df_prj,
        output_folder= model_dir,
        prj_file_name=GSSHA_prj_name
    )


    gf.copy_gssha_apps_to_model(GSSHA_executables, model_dir)
    gf.run_gssha(model_dir, PROJECT_FILE = GSSHA_prj_name + ".prj")
    gf.cleanup_model_dir(model_dir)

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".dep",
        output_folder=calibration_one_results_dir,
        new_filename=GSSHA_prj_name +"_" + str(gauge_reading) + "_OUTPUT-timeseries-depth-m-" + str(cfs_flow_for_filenames) + ".dep"
    )

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".gfl",
        output_folder=calibration_one_results_dir,
        new_filename=GSSHA_prj_name+"_" + str(gauge_reading) + "_OUTPUT-maxflood-dep-m-" + str(cfs_flow_for_filenames) + ".gfl"
    )
    
    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".oqc",
        output_folder=calibration_one_results_dir,
        new_filename=GSSHA_prj_name +"_" + str(gauge_reading) + "_OUTPUT_USGS-STATS-location-cms-" + str(cfs_flow_for_filenames) + ".oqc"
    )
    
    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".ows",
        output_folder=calibration_one_results_dir,
        new_filename=GSSHA_prj_name +"_" + str(gauge_reading) + "_OUTPUT_WSE-active-USGS-gauge-m-" + str(cfs_flow_for_filenames) + ".ows"
    )


✔️ Cleaned up model folder.
File containing 26 already exists. Skipping...
File containing 27 already exists. Skipping...
File containing 28 already exists. Skipping...
File containing 29 already exists. Skipping...
File containing 30 already exists. Skipping...
File containing 31 already exists. Skipping...
File containing 32 already exists. Skipping...
File containing 33 already exists. Skipping...
File containing 34 already exists. Skipping...


# 3.0 Use the initial stage map model output to fine tune the stage predictions further

In [20]:
### Determine model inputs for second round of calibration
calibration_one_results_dir

WindowsPath('C:/Users/bgorberg/Documents/GitHub/Flood_Stage_Maps/RUN_GSSHA/Waialua_FIM_testing/CALIBRATION_STAGE_MAPS_WITH_OFFSET_13.835')

For the purpose of this study, we want maps to the 100 year streamflow threshold. We will add 10 feet to the maximum flood stage in an effort to get up to the 100 year threshold. **NOT ALL STAGES WILL BE MODELED, only stages leading up to the 100 year threshold** the next model runs cap off once the gauge's discharge is = or > than the 100 year predictions. There is no need to determine the approx stage of the 100 year threshold, the model run code will do that for you

In [21]:

Max_flood_stage = Max_flood_stage + 10

Create a dataframe to store results. we will use power law curves to derive connections between the input flows and gauge levels in the model. Then we will evaluate convergence thresholds to ensure run time is not excessively long. 

In [22]:
#SET UP CALIBRATION FROM INITIAL INPUT FLOWS

# Once Offset is determined, set up model paramters for running stage height maps
gage_readings = np.arange(Min_flood_stage, Max_flood_stage + 1, 1).tolist()

#Set up the storage of results
modeling_calibration_results= {"gauge_readings_ft":[],
                            "expected_WSE_m":[],
                            "expected_WSE_ft":[],  
                            "convergence_time": [], 
                            "input_flows":[]}

#Fill in some metadata about gauge readings, expected model outputs to figure out the rest
expected_model_outputs = [x - Gauge_Datum_offset for x in gage_readings]
modeling_calibration_results["gauge_readings_ft"] = gage_readings
modeling_calibration_results["expected_WSE_ft"] = expected_model_outputs
modeling_calibration_results["expected_WSE_m"]  = [round(ft * 0.3048, 2) for ft in expected_model_outputs]

MODEL INPUT FLOW FOR EXPECTED WSE

In [23]:
# modeling_calibration_results["input_flows"] = gppf.get_input_flows_for_gauge_stages(
#     modeling_results_directory = calibration_one_results_dir,
#     requested_WSE =  modeling_calibration_results["expected_WSE_m"],
#     plot=True,
# )

read in the test flow model data and the initial flood stage maps

In [28]:
modeling_results_directory = [
    calibration_one_results_dir,
    test_input_flow_results_dir,
]
peak_wse = gppf.read_test_and_first_calibration_results (modeling_results_directory)
gppf.plot_power_log_function(
    peak_wse,
)

@plot and manually remove outlier

In [29]:
outliers = [50, 70, 79, 80, 90, 100, 111]

peak_wse_no_outlier = {
    key: value
    for key, value in peak_wse.items()
    if key not in outliers
}

get the interpolation variables for low flow

In [54]:
interpolation_variables = gppf.low_flow_interpolation(
    peak_wse_no_outlier,
    smoothing=0.01,
    plot=True,
)

KeyboardInterrupt: 

get the linear variables for high flow

In [31]:
linear_variables = gppf.high_flow_linear(
    peak_wse_no_outlier,
    above_value=1000, #cfs (where the curve turns linear)
    plot=True,
)

In [32]:
modeling_calibration_results["input_flows"] = (
    gppf.get_input_flow_interpolation_linear(
        expected_WSE_m=modeling_calibration_results[
            "expected_WSE_m"
        ],
        interpolation_variables=interpolation_variables,
        linear_variables=linear_variables,
        flow_threshold=1000,#what cms we wnat linear equation to take over, PCHIP use before high flow threshold
    )
)

MODEL TIMESTEP CONVERGENCE (YOU CHANGE PARAMETERS BASED ON GRAPH)

In [53]:
timestep_function = gppf.view_timestep_convergence(
    model_results_directory=calibration_one_results_dir,
    area_change_threshold=1000,
    number_timesteps_below_threshold=6,
    smoothing=0.025,
)

modeling_calibration_results["convergence_time"] = gppf.calculate_convergence_time(additional_buffer = 30, timestep_function = timestep_function, listed_input_flows = modeling_calibration_results['input_flows'] )

In [46]:
#save the model input parameters (from the previous section, derived from test flow inputs)
final_stage_model_input = pd.DataFrame(modeling_calibration_results).set_index('gauge_readings_ft')

# 3.1 Run model until it reaches the gauge's 100 year streamflow threshold

Use USGS Stats to find the streamflow amount (convert that to cms). IF USGS Stats doesn't have the value reported, you can try using PeakFQ from their website

https://rconnect.usgs.gov/peakfq/

In [83]:
#save model input results in folder for final stage maps
final_stage_maps_results_dir = model_dir / ("FINAL_STAGE_MAPS_WITH_OFFSET_" + str(Gauge_Datum_offset))
final_stage_maps_results_dir.mkdir(parents=True, exist_ok=True)

final_stage_model_input.to_csv(calibration_one_results_dir / "final_stage_maps_model_inputs.csv")
final_stage_maps_results_dir

WindowsPath('C:/Users/bgorberg/Documents/GitHub/Flood_Stage_Maps/RUN_GSSHA/Waialua_FIM_testing/FINAL_STAGE_MAPS_WITH_OFFSET_13.835')

In [82]:
one_hundred_year_threshold = 19690 #from peak fq
one_hundred_year_threshold = np.floor(one_hundred_year_threshold* 0.028316846592) #must be in CMS : cfs * 0.028316846592
one_hundred_year_threshold

557.0

In [49]:
final_stage_model_input

,expected_WSE_m,expected_WSE_ft,convergence_time,input_flows
gauge_readings_ft,,,,
26,3.71,12.165,753,113.828961
27,4.01,13.165,708,141.588992
28,4.32,14.165,655,177.214209
29,4.62,15.165,589,228.309226
30,4.93,16.165,503,318.077940
31,5.23,17.165,450,419.984705
32,5.54,18.165,411,562.249348
33,5.84,19.165,360,783.124243
34,6.15,20.165,294,1229.446171


In [ ]:
gf.cleanup_model_dir(model_dir)

for gauge_reading in final_stage_model_inputt.index:
    input_flow = final_stage_model_input['input_flows'][gauge_reading].round(0)
    total_run_time = final_stage_model_input['convergence_time'][gauge_reading]

    
    new_value = float(input_flow)
    cfs_flow_for_filenames = round(input_flow*35.31467)
    
    output_file = (
        final_stage_maps_results_dir
        / f"{GSSHA_prj_name}_{gauge_reading}_OUTPUT-input-flow-cfs-{cfs_flow_for_filenames}.tsf"
    )
    
    if any(
        str(gauge_reading) in file.name
        for file in final_stage_maps_results_dir .iterdir()
    ):
        print(f"File containing {gauge_reading} already exists. Skipping...")
        continue

    
    df_prj = gf.read_prj_file(GSSHA_prj_name +".prj", prj_folder_path = model_dir)
    

    #ITERATION FILES
    gf.replace_value_in_gssha_file(read_dir=model_dir , gssha_sample_file = xys_tsf_iteration_file_name + ".tsf", save_filename = GSSHA_prj_name + "_bc.tsf", old_value =xys_tsf_iteration_value, new_value = new_value)
    gf.replace_value_in_gssha_file(read_dir= model_dir, gssha_sample_file = xys_tsf_iteration_file_name + ".xys", save_filename = GSSHA_prj_name + ".xys", old_value = xys_tsf_iteration_value, new_value = new_value)
    ##UPDATE THE ITERATION W THE NEW XYS AND TSF FILE

    #!!!!write a function to copy with specified cfs flows in name!!!

    #SAVE TEXT XYS file for knowing the input flows for model run
    #These files are for the 

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + "_bc.tsf",
        output_folder=final_stage_maps_results_dir,
        new_filename=GSSHA_prj_name +"_" + str(gauge_reading) + "_OUTPUT-input-flow-cfs-" + str(cfs_flow_for_filenames) + ".tsf"
    )


    

    df_prj.loc['TOT_TIME', 'Value'] = total_run_time
    
    df_prj = df_prj.reset_index()


    #UPDATE PRJ FILE WHICH CONTAINS THE PATHS
    gf.convert_df_to_prj(
        df_prj,
        output_folder= model_dir,
        prj_file_name=GSSHA_prj_name
    )


    gf.copy_gssha_apps_to_model(GSSHA_executables, model_dir)
    gf.run_gssha(model_dir, PROJECT_FILE = GSSHA_prj_name + ".prj")
    gf.cleanup_model_dir(model_dir)

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".dep",
        output_folder=final_stage_maps_results_dir ,
        new_filename=GSSHA_prj_name +"_" + str(gauge_reading) + "_OUTPUT-timeseries-depth-m-" + str(cfs_flow_for_filenames) + ".dep"
    )

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".gfl",
        output_folder=final_stage_maps_results_dir ,
        new_filename=GSSHA_prj_name+"_" + str(gauge_reading) + "_OUTPUT-maxflood-dep-m-" + str(cfs_flow_for_filenames) + ".gfl"
    )
    
    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".oqc",
        output_folder=final_stage_maps_results_dir ,
        new_filename=GSSHA_prj_name +"_" + str(gauge_reading) + "_OUTPUT_USGS-STATS-location-cms-" + str(cfs_flow_for_filenames) + ".oqc"
    )
    
    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".ows",
        output_folder=final_stage_maps_results_dir,
        new_filename=GSSHA_prj_name +"_" + str(gauge_reading) + "_OUTPUT_WSE-active-USGS-gauge-m-" + str(cfs_flow_for_filenames) + ".ows"
    )

    
    oqc_filepath = model_dir / (GSSHA_prj_name + ".oqc")
    gauge_discharge = gppf.read_GSSHA_oqc(oqc_filepath)
    max_discharge = np.ceil(gauge_discharge["instantaneous_discharge"].max())
    
    if max_discharge >=  one_hundred_year_threshold:
        break


In [76]:
oqc_filepath = model_dir / (GSSHA_prj_name + ".oqc")
gauge_discharge = gppf.read_GSSHA_oqc(oqc_filepath)
max_discharge = np.ceil(gauge_discharge["instantaneous_discharge"].max())